# stochastic-rs — paths on the CPU and on CUDA, side by side

Samples a gallery of processes twice — once on the host, once on the GPU — and
plots the two next to each other, one figure per family: Brownian motion,
diffusions bounded and unbounded, short rates, stochastic volatility, jumps,
point processes, fractional processes, subordinators and the
conditional-variance time series. Seventy-odd rows, and every parameter set is
one the crate's own device-law suite runs, so a row that misbehaves here is a
finding rather than a typo.

**What is and is not comparable.** The host draws from this crate's SIMD
ziggurat and the kernel hashes its own normals from `(path, step, seed)`, so
the two never produce the same *path* — comparing them point for point is
meaningless. What must agree is the **law**: the third panel of each row
overlays the two terminal distributions, and the table at the end puts the
means and spreads beside each other. A row whose paths look different but
whose histograms sit on top of each other is exactly right.

**Where a process falls back.** Some configurations exceed what the kernels
carry, and the process says so: `device_fallback()` returns the reason and
`device_ready()` is the absence of one. Cell 5 prints both, and it also drops
a row this runtime cannot build at all — the curve-driven short rates are
f64-only, so on a single-precision device they say so and the rest of the
gallery goes on without them.

**The knobs, all in cell 5.** `DEVICE` names the device side (`"cuda"` here,
`"metal"` on a Mac), `DTYPE` forces a precision, `ONLY` restricts the run to a
few families, and `N`/`PATHS` set the grid and the batch.

Cells: 1 GPU · 2 Rust · 3 repository · 4 wheel (20-30 min) · 5 gallery ·
6 plots · 7 law table · 8 timings.

In [ ]:
# 1. The GPU and the CUDA toolkit. The driver's CUDA version (nvidia-smi, top right)
#    should be >= the toolkit's (nvcc): NVRTC emits PTX for the toolkit's version and
#    an older driver cannot JIT it (CUDA_ERROR_UNSUPPORTED_PTX_VERSION).
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
# 2. Rust (stable, minimal profile). PATH is extended for every later cell.
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CARGO_TERM_COLOR"] = "never"
!cargo --version && rustc --version

In [ ]:
# 3. The repository. REF is a branch, tag or commit; main is the default.
REF = "main"
!rm -rf stochastic-rs && git clone --quiet --depth 1 --branch {REF} https://github.com/rust-dd/stochastic-rs.git
%cd stochastic-rs
!git log --oneline -1

In [ ]:
# 4. The Python module with the CUDA back-end. Release build, 20-30 minutes on
#    Colab; maturin needs an explicit interpreter because Colab has no venv.
!pip -q install maturin numpy matplotlib
!maturin build --release --features cuda --interpreter python3 --out dist 2>&1 | tail -2
!pip -q install --force-reinstall --no-deps dist/*.whl

import importlib
import stochastic_rs as sr
sr = importlib.reload(sr)
print(sr.probe_device("cuda"))
print(sr.probe_device("cpu"))

In [ ]:
# 5. The gallery: what to sample, which component to plot, and what the crate
#    says about each configuration before anything runs. Seventy-odd processes
#    across ten families; every parameter set here is one the crate's own
#    device-law suite runs, so a row that misbehaves is a finding rather than a
#    typo. Set ONLY to a few family names to run part of the gallery.
from typing import Callable, NamedTuple

import numpy as np
import stochastic_rs as sr

DEVICE = "cuda"  # the device side of every comparison; "metal" on a Mac
N = 512          # grid points per path
PATHS = 4_000    # paths per side; the plots draw the first few
DRAWN = 24       # paths per panel
J = 64           # series terms for the truncation-series Levy processes
SEED = 20260907
ONLY = ()        # e.g. ("volatility", "jump") to run part of the gallery
DTYPE = "f32"    # the precision both sides compute in; None keeps each class's f64


def dev(d):
  """The seed, the device and the precision every builder passes.

  `f32` is what a comparison should use on a consumer NVIDIA card, which runs
  double precision at a fraction of its single-precision rate, and it is the
  only precision Metal has. The five curve-driven short rates below take no
  `dtype` at all — a Python callable coefficient is called in double
  precision, so those classes have no `f32` slot — and they pass the seed and
  the device on their own.
  """
  kw = {"seed": SEED, "device": d}
  if DTYPE:
    kw["dtype"] = DTYPE
  return kw


class Row(NamedTuple):
  """One gallery entry: a builder taking the device name, and what to plot."""

  family: str
  label: str
  build: Callable[[str], object]
  comp: int = 0   # which component of a multi-component process to plot
  at: float = 1.0  # the point whose law is compared, as a fraction of the grid


def component(x, i=0):
  """One (paths, points) block out of whatever shape a process returns: the
  i-th array of a tuple, the i-th slot of a stacked axis, the real (i=0) or
  imaginary (i=1) part of a complex-valued process, and a list of per-path
  arrays stacked back into one."""
  if isinstance(x, tuple):
    x, i = x[i], 0
  if isinstance(x, list):
    x = np.stack([np.asarray(path) for path in x])
  x = np.asarray(x)
  if x.ndim == 3:
    x, i = x[:, i, :], 0
  if np.iscomplexobj(x):
    x = x.real if i == 0 else x.imag
  return np.asarray(x, dtype=float)


def curve(t):
  """A rising instantaneous-forward curve, for the models that take one."""
  return 0.02 + 0.03 * t


GALLERY = [
  Row("brownian", "Brownian motion",
      lambda d: sr.PyBm(N, t=1.0, **dev(d))),
  # A bridge is pinned at both ends, so its terminal value is the same number
  # on every path and in both samples; the midpoint is where it has a law.
  Row("brownian", "Brownian bridge  sigma=0.3  0 -> 1",
      lambda d: sr.PyBrownianBridge(0.3, N, x0=0.0, xt=1.0, t=1.0, **dev(d)), at=0.5),
  Row("brownian", "correlated BM pair  rho=-0.5  (leg 1)",
      lambda d: sr.PyCbms(-0.5, N, t=1.0, **dev(d)), 0),
  Row("brownian", "correlated BM pair  rho=-0.5  (leg 2)",
      lambda d: sr.PyCbms(-0.5, N, t=1.0, **dev(d)), 1),

  Row("diffusion", "GBM  mu=0.05 sigma=0.2",
      lambda d: sr.PyGbm(0.05, 0.2, N, x0=100.0, t=1.0, **dev(d))),
  Row("diffusion", "GBM in log space  mu=0.05",
      lambda d: sr.PyGbmLog(0.05, sigma=0.2, n=N, s0=100.0, t=1.0, **dev(d))),
  Row("diffusion", "Ornstein-Uhlenbeck  theta=2 mu=0.04",
      lambda d: sr.PyOu(2.0, 0.04, 0.25, N, x0=0.12, t=1.0, **dev(d))),
  Row("diffusion", "CIR  theta=2 mu=0.04",
      lambda d: sr.PyCir(2.0, 0.04, 0.2, N, x0=0.04, t=1.0, **dev(d))),
  Row("diffusion", "CEV  gamma=0.8",
      lambda d: sr.PyCev(0.05, 0.2, 0.8, N, x0=100.0, t=1.0, **dev(d))),
  Row("diffusion", "CKLS  theta=(0.06,-1.5,0.3,0.5)",
      lambda d: sr.PyCkls(0.06, -1.5, 0.3, 0.5, N, x0=0.04, t=1.0, **dev(d))),
  Row("diffusion", "3/2 model  kappa=2 mu=0.04",
      lambda d: sr.PyThreeHalf(2.0, 0.04, 0.3, N, x0=0.04, t=1.0, **dev(d))),
  Row("diffusion", "squared Bessel  delta=3",
      lambda d: sr.PySquaredBessel(3.0, N, x0=1.0, t=1.0, **dev(d))),
  Row("diffusion", "displaced diffusion  beta=20",
      lambda d: sr.PyDisplacedDiffusion(0.05, 0.2, 20.0, N, x0=100.0, t=1.0, **dev(d))),
  Row("diffusion", "linear SDE  a=0.02 b=0.3",
      lambda d: sr.PyLinearSDE(0.02, 0.3, 0.2, N, x0=1.0, t=1.0, **dev(d))),
  Row("diffusion", "radial OU  kappa=1",
      lambda d: sr.PyRadialOU(1.0, 0.3, N, x0=1.0, t=1.0, **dev(d))),
  Row("diffusion", "Ait-Sahalia  (short rate, nonlinear drift)",
      lambda d: sr.PyAitSahalia(0.0001, 0.15, -3.0, 0.0, 0.0004, 0.0, 0.05, 1.5, N,
                                x0=0.05, t=1.0, **dev(d))),

  Row("bounded diffusion", "Jacobi  alpha=0.3 beta=0.6  (unit interval)",
      lambda d: sr.PyJacobi(0.3, 0.6, 0.2, N, x0=0.5, t=1.0, **dev(d))),
  Row("bounded diffusion", "Kimura  a=0.5  (unit interval)",
      lambda d: sr.PyKimura(0.5, 0.2, N, x0=0.5, t=1.0, **dev(d))),
  Row("bounded diffusion", "Verhulst  r=1 K=2  (clamped)",
      lambda d: sr.PyVerhulst(1.0, 2.0, 0.3, N, x0=0.5, t=1.0, clamp=True, **dev(d))),
  Row("bounded diffusion", "Feller root  theta=(0.5,0.3,0.2)",
      lambda d: sr.PyFellerRoot(0.5, 0.3, 0.2, N, x0=0.5, t=1.0, **dev(d))),
  Row("bounded diffusion", "Feller logistic  kappa=1 theta=1",
      lambda d: sr.PyFellerLogistic(1.0, 1.0, 0.3, N, x0=0.5, t=1.0, use_sym=False,
                                    **dev(d))),
  Row("bounded diffusion", "hyperbolic  kappa=1",
      lambda d: sr.PyHyperbolic(1.0, 0.3, N, x0=0.5, t=1.0, **dev(d))),
  Row("bounded diffusion", "Pearson  kappa=1 mu=0.3",
      lambda d: sr.PyPearson(1.0, 0.3, 0.0, 0.0, 0.01, N, x0=0.3, t=1.0, **dev(d))),
  Row("bounded diffusion", "Gompertz  a=0.5 b=0.3",
      lambda d: sr.PyGompertz(0.5, 0.3, 0.2, N, x0=1.0, t=1.0, **dev(d))),
  Row("bounded diffusion", "Teng stochastic correlation  (-1,1)",
      lambda d: sr.PyTengSCP(1.0, 0.3, 0.4, 0.2, N, t=1.0, **dev(d))),

  Row("short rate", "Vasicek  theta=0.5 mu=0.04",
      lambda d: sr.PyVasicek(0.5, 0.04, 0.02, N, x0=0.03, t=1.0, **dev(d))),
  Row("short rate", "fractional Vasicek  H=0.7",
      lambda d: sr.PyFVasicek(0.7, 2.0, 0.04, 0.02, N, x0=0.03, t=1.0, **dev(d))),
  Row("short rate", "Ho-Lee  theta=0.03 sigma=0.01",
      lambda d: sr.PyHoLee(0.01, N, theta=0.03, t=1.0, seed=SEED, device=d)),
  Row("short rate", "Hull-White  (curve from Python)",
      lambda d: sr.PyHullWhite(curve, 1.0, 0.01, N, x0=0.02, t=1.0, seed=SEED, device=d)),
  Row("short rate", "Hull-White 2F  rho=-0.4",
      lambda d: sr.PyHullWhite2F(curve, 1.0, 0.01, 0.005, -0.4, 0.5, N, x0=0.02, t=1.0,
                                 seed=SEED, device=d), 0),
  Row("short rate", "Black-Karasinski  (lognormal short rate)",
      lambda d: sr.PyBlackKarasinski(curve, 1.0, 0.2, N, r0=0.03, t=1.0, seed=SEED, device=d)),
  Row("short rate", "CIR++  (CIR plus a deterministic shift)",
      lambda d: sr.PyCirPlusPlus(2.0, 0.04, 0.2, curve, N, x0=0.04, t=1.0, use_sym=False,
                                 seed=SEED, device=d)),
  Row("short rate", "Duffie-Kan  (short rate leg)",
      lambda d: sr.PyDuffieKan(0.5, 0.2, 0.1, -0.3, -0.5, 0.1, 0.02, 0.1, 0.05, -0.3, 0.01,
                               0.08, N, r0=0.03, x0=0.01, t=1.0, **dev(d)), 0),
  Row("short rate", "Duffie-Kan with exponential jumps",
      lambda d: sr.PyDuffieKanJumpExp(0.5, 0.2, 0.1, -0.3, -0.5, 0.1, 0.02, 0.1, 0.05, -0.3,
                                      0.01, 0.08, 3.0, 0.01, N, r0=0.03, x0=0.01, t=1.0,
                                      **dev(d)), 0),

  Row("volatility", "Heston  kappa=2 xi=0.3 rho=-0.7  (price)",
      lambda d: sr.PyHeston(2.0, 0.04, 0.3, -0.7, 0.0, N, s0=100.0, v0=0.04, t=1.0,
                            use_sym=False, **dev(d)), 0),
  Row("volatility", "Heston  (variance)",
      lambda d: sr.PyHeston(2.0, 0.04, 0.3, -0.7, 0.0, N, s0=100.0, v0=0.04, t=1.0,
                            use_sym=False, **dev(d)), 1),
  Row("volatility", "Heston in log space  mu=0.03",
      lambda d: sr.PyHestonLog(mu=0.03, kappa=2.0, theta=0.04, xi=0.3, rho=-0.7, n=N,
                               s0=100.0, v0=0.04, t=1.0, **dev(d)), 0),
  Row("volatility", "SABR  alpha=0.4 beta=0.5  (forward)",
      lambda d: sr.PySabr(0.4, 0.5, -0.4, N, f0=100.0, v0=0.2, t=1.0, **dev(d)), 0),
  Row("volatility", "SABR  (volatility)",
      lambda d: sr.PySabr(0.4, 0.5, -0.4, N, f0=100.0, v0=0.2, t=1.0, **dev(d)), 1),
  Row("volatility", "Bergomi  nu=0.5 rho=-0.6",
      lambda d: sr.PyBergomi(0.5, 0.02, -0.6, N, v0=0.2, s0=100.0, t=1.0, **dev(d)), 0),
  Row("volatility", "Bates SVJ  lambda=3",
      lambda d: sr.PyBatesSvj(3.0, -0.05, 0.1, 0.08, 2.0, 0.3, -0.7, N, mu=0.02, s0=100.0,
                              v0=0.04, t=1.0, use_sym=False, **dev(d)), 0),
  Row("volatility", "Fouque OU 2F  (fast-slow volatility)",
      lambda d: sr.PyFouqueOU2D(1.0, 0.3, 0.25, -0.2, N, x0=0.0, y0=0.0, t=1.0,
                                **dev(d)), 0),
  Row("volatility", "SVCGMY  (CGMY with stochastic clock)",
      lambda d: sr.PySvcgmy(2.0, 6.0, 0.5, 2.0, 0.04, 0.2, 0.3, N, J, x0=0.0, v0=0.04, t=1.0,
                            **dev(d)), 0),

  Row("jump", "Merton in log space  lambda=3",
      lambda d: sr.PyMjdLog(0.05, sigma=0.2, lambda_=3.0, nu=-0.05, omega=0.1, n=N, s0=100.0,
                            t=1.0, **dev(d))),
  Row("jump", "variance gamma  theta=-0.1 nu=0.5",
      lambda d: sr.PyVg(-0.1, 0.2, 0.5, N, x0=0.0, t=1.0, **dev(d))),
  Row("jump", "normal inverse Gaussian  kappa=0.5",
      lambda d: sr.PyNig(-0.1, 0.2, 0.5, N, x0=0.0, t=1.0, **dev(d))),
  Row("jump", "CGMY  Y=0.5",
      lambda d: sr.PyCgmy(1.0, 2.0, 6.0, 0.5, N, J, x0=0.0, t=1.0, **dev(d))),
  Row("jump", "KoBoL  alpha=0.5",
      lambda d: sr.PyKoBoL(1.0, 2.0, 1.0, 3.0, 3.0, 0.5, N, J, x0=0.0, t=1.0, **dev(d))),
  Row("jump", "classical tempered stable  alpha=0.5",
      lambda d: sr.PyCts(2.0, 6.0, 0.5, N, J, x0=0.5, t=1.0, **dev(d))),
  Row("jump", "bilateral gamma motion",
      lambda d: sr.PyBilateralGammaMotion(0.1, 1.5, 10.0, 1.2, 12.0, N, x0=0.0, t=1.0,
                                          **dev(d))),
  Row("jump", "Hawkes jump-diffusion  (self-exciting)",
      lambda d: sr.PyHawkesJD(0.02, 0.2, 1.0, 0.5, 2.0, -0.02, 0.05, N, x0=0.0, t=1.0,
                              **dev(d))),
  Row("jump", "inverse Gaussian motion  gamma=1",
      lambda d: sr.PyIg(1.0, N, x0=0.0, t=1.0, **dev(d))),

  Row("point process", "Poisson  lambda=25  (arrival times)",
      lambda d: sr.PyPoisson(25.0, n=N, **dev(d))),
  Row("point process", "Hawkes  mu=1 alpha=0.5 beta=1.5  (event times)",
      lambda d: sr.PyHawkes(1.0, 0.5, 1.5, n=64, **dev(d))),

  Row("fractional", "fBm  H=0.7",
      lambda d: sr.PyFbm(0.7, N, t=1.0, **dev(d))),
  Row("fractional", "fGN  H=0.3  (increments)",
      lambda d: sr.PyFgn(0.3, N, t=1.0, **dev(d))),
  Row("fractional", "fractional OU  H=0.7",
      lambda d: sr.PyFou(0.7, 2.0, 1.0, 0.3, N, x0=0.0, t=1.0, **dev(d))),
  Row("fractional", "fractional GBM  H=0.7",
      lambda d: sr.PyFgbm(0.7, 0.05, 0.2, N, x0=100.0, t=1.0, **dev(d))),
  Row("fractional", "fractional CIR  H=0.7",
      lambda d: sr.PyFcir(0.7, 2.0, 0.04, 0.1, N, x0=0.04, t=1.0, **dev(d))),
  Row("fractional", "fractional Jacobi  H=0.7",
      lambda d: sr.PyFJacobi(0.7, 0.3, 0.6, 0.2, N, x0=0.5, t=1.0, **dev(d))),
  Row("fractional", "Levy fractional stable motion  alpha=1.7 H=0.7",
      lambda d: sr.PyLfsm(1.7, 0.2, 0.7, 0.1, N, x0=0.0, t=1.0, **dev(d))),
  Row("fractional", "correlated fBm pair  H=0.7 rho=0.4  (leg 1)",
      lambda d: sr.PyCfbms(0.7, 0.4, N, t=1.0, **dev(d)), 0),
  Row("fractional", "correlated fBm pair  (leg 2)",
      lambda d: sr.PyCfbms(0.7, 0.4, N, t=1.0, **dev(d)), 1),
  Row("fractional", "complex fOU  H=0.7  (real part)",
      lambda d: sr.PyCfou(0.7, 2.0, 1.5, 0.6, N, x1_0=0.1, x2_0=-0.1, t=1.0,
                          **dev(d)), 0),
  Row("fractional", "complex fOU  (imaginary part)",
      lambda d: sr.PyCfou(0.7, 2.0, 1.5, 0.6, N, x1_0=0.1, x2_0=-0.1, t=1.0,
                          **dev(d)), 1),

  Row("subordinator", "alpha-stable  alpha=0.7",
      lambda d: sr.PyAlphaStableSubordinator(0.7, 1.0, N, x0=0.0, t=1.0, **dev(d))),
  Row("subordinator", "gamma  nu=2 rate=1.5",
      lambda d: sr.PyGammaSubordinator(2.0, 1.5, N, x0=0.0, t=1.0, **dev(d))),
  Row("subordinator", "inverse Gaussian  delta=1 gamma=2",
      lambda d: sr.PyIGSubordinator(1.0, 2.0, N, x0=0.0, t=1.0, **dev(d))),
  Row("subordinator", "tempered stable  alpha=0.6",
      lambda d: sr.PyTemperedStableSubordinator(0.6, 1.0, 2.0, 0.05, N, x0=0.0, t=1.0,
                                                **dev(d))),
  Row("subordinator", "Poisson  lambda=20  (a counting clock)",
      lambda d: sr.PyPoissonSubordinator(20.0, N, x0=0.0, t=1.0, **dev(d))),
  Row("subordinator", "inverse alpha-stable  alpha=0.7  (a waiting clock)",
      lambda d: sr.PyInverseAlphaStableSubordinator(0.7, 1.0, N, t=1.0, u_steps=256,
                                                    **dev(d))),

  Row("time series", "AR(1)  phi=0.6",
      lambda d: sr.PyARp([0.6], 0.2, N, x0=[0.5], **dev(d))),
  Row("time series", "MA(1)  theta=0.4",
      lambda d: sr.PyMAq([0.4], 0.2, N, **dev(d))),
  Row("time series", "ARIMA(2,1,1)",
      lambda d: sr.PyArima([0.6, -0.2], [0.3], 1, 0.5, N, **dev(d))),
  Row("time series", "ARCH(1)  alpha=0.3",
      lambda d: sr.PyArch(0.0002, [0.3], N, **dev(d))),
  Row("time series", "GARCH(1,1)  alpha=0.1 beta=0.85",
      lambda d: sr.PyGarch(0.00001, [0.1], [0.85], N, **dev(d))),
  Row("time series", "EGARCH(1,1)  (log variance, leverage)",
      lambda d: sr.PyEgarch(-0.2, [0.1], [-0.05], [0.95], N, **dev(d))),
  Row("time series", "GJR-GARCH(1,1)  (threshold leverage)",
      lambda d: sr.PyGjrGarch(0.00001, [0.05], [0.1], [0.85], N, **dev(d))),
]

if ONLY:
  GALLERY = [row for row in GALLERY if row.family in ONLY]
FAMILIES = list(dict.fromkeys(row.family for row in GALLERY))

# What each process says about itself before anything runs: whether a kernel
# carries this configuration, and the reason when it does not. Each row is
# also sampled four paths deep on both sides, and one that cannot be built
# here is dropped with its reason rather than taking the whole cell down —
# the curve-driven short rates, for one, are f64-only and cannot run on a
# single-precision device.
print(f"{len(GALLERY)} rows over {len(FAMILIES)} families, device = {DEVICE!r}\n")
print(f"{'process':52s} {'runs on':8s} reason")
usable = []
for family in FAMILIES:
  print(f"-- {family}")
  for row in (r for r in GALLERY if r.family == family):
    try:
      p = row.build(DEVICE)
      shapes = {component(row.build("cpu").sample_par(4), row.comp).shape,
                component(p.sample_par(4), row.comp).shape}
      if len(shapes) != 1 or len(next(iter(shapes))) != 2:
        raise ValueError(f"not one (paths, points) block on both sides: {shapes}")
    except Exception as e:
      print(f"   {row.label:49s} {'dropped':8s} {type(e).__name__}: {e}")
      continue
    usable.append(row)
    print(f"   {row.label:49s} {'kernel' if p.device_ready() else 'host':8s} "
          f"{p.device_fallback() or ''}")

GALLERY = usable
FAMILIES = [f for f in FAMILIES if any(r.family == f for r in GALLERY)]

In [ ]:
# 6. The plots, one figure per family: host paths, device paths, and the two
#    terminal laws overlaid. Left and centre share a y-scale so the eye
#    compares dispersion rather than axes, and both that scale and the
#    histogram's range are the central 99.8% of the pooled sample — one
#    excursion of an alpha-stable path would otherwise flatten everything
#    else in the row. Only the terminal column of each sample is kept, so the
#    whole gallery costs a few megabytes rather than a few gigabytes.
import matplotlib.pyplot as plt

compared = {}


def sample_both(row):
  host = component(row.build("cpu").sample_par(PATHS), row.comp)
  device = component(row.build(DEVICE).sample_par(PATHS), row.comp)
  return host, device


def span(*arrays, lo=0.1, hi=99.9):
  pool = np.concatenate([a.ravel() for a in arrays])
  a, b = np.percentile(pool, [lo, hi])
  pad = 0.05 * (b - a) if b > a else 1.0
  return a - pad, b + pad


for family in FAMILIES:
  rows = [r for r in GALLERY if r.family == family]
  fig, axes = plt.subplots(len(rows), 3, figsize=(15, 2.6 * len(rows)), squeeze=False)
  fig.suptitle(f"{family} — host, {DEVICE}, and the terminal law of each",
               y=1.0, fontsize=13)
  for i, row in enumerate(rows):
    host, device = sample_both(row)
    col = round(row.at * (host.shape[1] - 1))
    compared[row.label] = (host[:, col].copy(), device[:, col].copy())
    grid = np.arange(host.shape[1])
    lo, hi = span(host[:DRAWN], device[:DRAWN])
    for col, (paths, name) in enumerate(((host, "CPU"), (device, DEVICE.upper()))):
      ax = axes[i][col]
      ax.plot(grid, paths[:DRAWN].T, lw=0.6, alpha=0.75)
      ax.set_ylim(lo, hi)
      ax.set_title(f"{row.label} — {name}" if col == 0 else name, fontsize=9, loc="left")
      ax.tick_params(labelsize=7)
    ax = axes[i][2]
    h, d = compared[row.label]
    edges = np.linspace(*span(h, d, lo=0.5, hi=99.5), 60)
    ax.hist(h, bins=edges, histtype="step", lw=1.2, label="CPU", density=True)
    ax.hist(d, bins=edges, histtype="step", lw=1.2, label=DEVICE.upper(), density=True)
    ax.set_title("terminal law" if row.at == 1.0 else f"law at {row.at:.0%} of the path",
                 fontsize=9, loc="left")
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, frameon=False)
  fig.tight_layout()
  plt.show()

In [ ]:
# 7. The law table. The paths differ by construction; these numbers must not.
#    The band is the standard error of the difference of two independent
#    sample means (and of two spreads), so it is what the sample size allows
#    rather than a tolerance anyone chose. The column compared is the terminal
#    one, except where a row says otherwise: a pinned process has no law at
#    its endpoint.


def spread_se(x):
  """The standard error of a sample standard deviation, sd*sqrt((k-1)/4n)."""
  m2 = x.var()
  if m2 == 0:
    return 0.0
  k = ((x - x.mean()) ** 4).mean() / (m2 * m2)
  return np.sqrt(m2) * np.sqrt(max(k - 1.0, 0.0) / (4 * len(x)))


def z_score(a, b, se):
  """A difference in standard errors, and 0 rather than 0/0 when both sides
  are degenerate — a Brownian bridge's terminal value is pinned, so its two
  samples have no spread to divide by and nothing to disagree about."""
  if se > 0:
    return (a - b) / se
  return 0.0 if a == b else np.inf


worst = []
print(f"{'process':52s} {'mean cpu':>12s} {'mean dev':>12s} {'z':>6s} "
      f"{'sd cpu':>11s} {'sd dev':>11s} {'z':>6s}")
for family in FAMILIES:
  print(f"-- {family}")
  for row in (r for r in GALLERY if r.family == family):
    h, d = compared[row.label]
    n = len(h)
    mean_z = z_score(h.mean(), d.mean(), np.sqrt(h.var(ddof=1) / n + d.var(ddof=1) / n))
    sd_z = z_score(h.std(), d.std(), np.hypot(spread_se(h), spread_se(d)))
    worst.append((max(abs(mean_z), abs(sd_z)), row.label))
    print(f"   {row.label:49s} {h.mean():12.5f} {d.mean():12.5f} {mean_z:6.1f} "
          f"{h.std():11.5f} {d.std():11.5f} {sd_z:6.1f}")

worst.sort(reverse=True)
print("\nz is in standard errors; |z| under 5 is agreement, and a process that "
      "fell back to the host\nhas z = 0 by construction. The five widest rows:")
for z, label in worst[:5]:
  print(f"   {z:6.1f}  {label}")
print("\nRead the heavy-tailed rows with care: an alpha-stable law has no mean, "
      "so its two sample\nmeans are whatever the largest jump in each sample "
      "happened to be, and the z column —\nwhich divides by the sample's own "
      "spread — is the honest comparison there.")

In [ ]:
# 8. Wall time per batch, host against device, for one process per family.
#    The GPU wins where the batch is large and the grid long; on a small batch
#    the launch dominates and the CPU is faster, which is worth seeing rather
#    than assuming.
import time


def timed(row, device, m):
  row.build(device).sample_par(m)      # warm the context, the kernel cache and
                                       # the output buffer, which the engine keeps
                                       # between launches and grows to the batch
  start = time.perf_counter()
  component(row.build(device).sample_par(m), row.comp)
  return (time.perf_counter() - start) * 1e3


print(f"{'process':52s} {'paths':>7s} {'cpu ms':>9s} {'dev ms':>9s} {'speed-up':>9s}")
for family in FAMILIES:
  row = next(r for r in GALLERY if r.family == family)
  for m in (256, 20_000):
    cpu_ms = timed(row, "cpu", m)
    dev_ms = timed(row, DEVICE, m)
    print(f"{row.label:52s} {m:7d} {cpu_ms:9.1f} {dev_ms:9.1f} {cpu_ms / dev_ms:8.1f}x")

## Reading the result

- **The path panels differ, and should.** Host and device draw different
  streams; only the law is shared. What to look for is the *shape* — the same
  drift, the same dispersion, the same boundary behaviour (a CIR that never
  goes negative, a Jacobi inside the unit interval, a subordinator that only
  increases).
- **The histograms should sit on top of each other**, and the table's `z`
  columns should stay inside ±5. A `z` of 30 is a real disagreement and worth
  an issue; a `z` of 4 on a heavy-tailed process (the α-stable subordinator,
  the 3/2 model) is the sample size talking. Cell 7 ends with the five widest
  rows, which is where to look first.
- **A row marked `host` in cell 5** ran on the CPU on both sides despite
  `device=`, for the reason printed beside it — its `z` values are then a
  comparison of the host with itself. A row marked `dropped` never ran: the
  class could not be built on this runtime, and the reason says why.
- **Cell 8's small-batch rows are usually red for the GPU.** One launch has a
  fixed cost; the device pays off from a few thousand paths, and the crate
  chunks a batch too large for the device's memory into launches whose union
  is bit-identical to one.